In [1]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser


llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    api_key=os.environ['GOOGLE_GEMINI_API_KEY'],
    temperature=0
)

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful AI assistant. "
        "Use the conversation history to understand the user."
    ),

    MessagesPlaceholder(
        variable_name="history"
    ),

    (
        "human",
        "{input}"
    )
])

chain = prompt | llm | StrOutputParser()

store = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)


def chat(session_id, user_input):

    response = chain_with_memory.invoke(
        {
            "input": user_input
        },

        config={
            "configurable": {
                "session_id": session_id
            }
        }
    )

    return response



response = chat(
    "user_1",
    "My name is Ashmi."
)

print("AI:", response)


response = chat(
    "user_1",
    "What is my name?"
)

print("AI:", response)


response = chat(
    "user_2",
    "My name is Rithi."
)

print("AI:", response)


response = chat(
    "user_2",
    "What is my name?"
)

print("AI:", response)


print("\n================ STORE ================\n")

for session_id, history in store.items():

    print("SESSION:", session_id)

    for message in history.messages:

        print(
            message.type,
            ":",
            message.content
        )

    print()

c:\Users\ashmi\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\ashmi\AppData\Local\Programs\Python\Python314\Lib\site-packages\IPython\core\interactiveshell.py:3747: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


AI: Hello Ashmi! It's great to meet you. How can I help you today?
AI: Your name is Ashmi!
AI: Hello Rithi! It's great to meet you. How can I help you today?
AI: Your name is Rithi! How can I help you today?

================ STORE ================

SESSION: user_1
human : My name is Ashmi.
ai : Hello Ashmi! It's great to meet you. How can I help you today?
human : What is my name?
ai : Your name is Ashmi!

SESSION: user_2
human : My name is Rithi.
ai : Hello Rithi! It's great to meet you. How can I help you today?
human : What is my name?
ai : Your name is Rithi! How can I help you today?

